<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import json
import pennylane as qml
import pennylane.numpy as np
from pennylane.ops import QubitStateVector
from pennylane.measurements import StateMeasurement


class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """Calculates <|M|> from the quantum state.

        Args:
            state (Sequence[complex]): The quantum state vector.
            wire_order (Wires): Wires determining the subspace of the state.

        Returns:
            float: <|M|>
        """
        num_wires = len(wire_order)
        N = num_wires
        abs_magnetization = 0.0

        for sigma in range(2 ** N):
            # Create |sigma> basis state
            sigma_state = np.zeros(2 ** N, dtype=complex)
            sigma_state[sigma] = 1.0

            # Probability of measuring |sigma>
            prob = np.abs(np.dot(np.conj(sigma_state), state)) ** 2

            # Compute sum of Z_i on |sigma>
            z_sum = 0
            for i in range(N):
                if (sigma >> i) & 1:
                    z_sum -= 1
                else:
                    z_sum += 1

            abs_magnetization += prob * abs(z_sum)

        return abs_magnetization / N


def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (numpy.tensor): The ground state.
    """
    coeffs = []
    observables = []

    # Interaction terms Z_i Z_{i+1}
    for i in range(num_qubits - 1):
        coeffs.append(-1.0)
        observables.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # Transverse field terms X_i
    for i in range(num_qubits):
        coeffs.append(-h)
        observables.append(qml.PauliX(i))

    H = qml.Hamiltonian(coeffs, observables)

    # Diagonalize to get ground state
    H_matrix = H.matrix(wire_order=range(num_qubits))
    eigvals, eigvecs = np.linalg.eigh(H_matrix)
    ground_state = eigvecs[:, 0]

    return ground_state


dev = qml.device("default.qubit", wires=lambda _: _)


@qml.qnode(dev)
def magnetization(num_qubits, h):
    """Calculates the absolute value of the magnetization of the 1D TFIM
    Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (float): <|M|>.
    """
    ground_state = tfim_ground_state(num_qubits, h)
    QubitStateVector(ground_state, wires=range(num_qubits))
    return AbsMagnetization(wires=list(range(num_qubits)))


def critical_point_estimate(mags, h_values):
    """Provides a finite-size estimate of the critical point of the 1D TFIM
    Hamiltonian. The estimate is done by taking the average value of h for which
    adjacent values of <|M|> differ the most.

    Args:
        mags (numpy.tensor):
            <|M|> values for various values of h (the transverse field strength).
        h_values (numpy.tensor): The transverse field strength values.

    Returns:
        (float): The critical point estimate, h_c.
    """
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))
    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c


# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []

    for h in h_values:
        mags.append(magnetization(num_qubits, h))

    output = critical_point_estimate(np.array(mags), h_values)

    return json.dumps(output)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.isclose(solution_output, expected_output, rtol=5e-3), f"Expected {expected_output}, got {solution_output}"


# These are the public test cases
test_cases = [
    ('5', '0.6735'),
    ('2', '0.3535')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError as exc:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


SyntaxError: invalid syntax (252837567.py, line 1)